In [ ]:
import lsdb
from dask.distributed import Client
import numpy as np
import matplotlib.pyplot as plt


import logging

In [ ]:
client=Client(n_workers=4, memory_limit="8GB", threads_per_worker=1, silence_logs=logging.ERROR)
client

## 1. Generate Comparison Catalog

In [ ]:
dp2_fitted = lsdb.open_catalog("../dp2_fitted_medium")
li_gaia = lsdb.open_catalog("/astro/store/epyc/data/gaia_rrl/li_xmatch/")


In [ ]:
li_gaia

In [ ]:
dp2_fitted.plot_pixels()
li_gaia.plot_pixels(alpha=0.4)

In [ ]:
li_gaia.columns

In [ ]:
dp2_fitted.columns

In [ ]:
# Getting bizarre column duplication errors even though the columns do not overlap, need suffixes like this
comparison_catalog = li_gaia.crossmatch(dp2_fitted, suffixes=("_","_dp2")) # inner match
comparison_catalog

In [ ]:
comparison_catalog.write_catalog("../dp2_gaia_comp_medium", overwrite=True)

## 2. Analyze Comparison Catalog

In [ ]:
comp = lsdb.open_catalog("../dp2_gaia_comp_medium")
comp_df = comp.compute()

In [ ]:
comp_df

In [ ]:
len(comp)

In [ ]:
comp.columns

### 2.1 p_est (pipeline) vs. Li+22 period 

In [ ]:
period_df = comp_df[["EBV_gaia_rrl_w_dist_","p_est_dp2", "p_err_dp2"]]
period_df

In [ ]:
import pandas as pd
des_df = pd.read_csv("/astro/store/epyc/data/des_rrl/des_rrab_catalog.csv")
plt.hist(des_df["PERIOD_0"])

In [ ]:
plt.hist(period_df["p_est_dp2"])


### 2.2 Compare Distances

mu from fit_coeffs is a proxy for distance — compare directly to Gaia parallax-derived distances or PL-based distances in Li+22


In [ ]:
distance_df = comp_df[['Dist_Phot_gaia_rrl_w_dist_','e_Dist_Phot_gaia_rrl_w_dist_',"mu_dp2"]]
distance_df["delta"] = distance_df["mu_dp2"] - distance_df["Dist_Phot_gaia_rrl_w_dist_"]
distance_df

In [ ]:
# Find fraction within 1 sigma
within_1sigma = np.abs(distance_df["delta"]) < distance_df["e_Dist_Phot_gaia_rrl_w_dist_"]
fraction_within_1sigma = np.mean(within_1sigma)
print(f"Fraction of stars with distance agreement within 1 sigma: {fraction_within_1sigma:.2f}")

In [ ]:
plt.hist(distance_df["delta"])

### 2.3 Check for Systematic Offsets

Per-band Δm offsets from fit_coeffs encode color — compare to Li+22 mean magnitudes per band to check for systematic photometric offsets between DP2 and GaiA.

In [ ]:
dp2_offset_columns = [f"offset_{band}_dp2" for band in ["u","g","r","i","z","y"]]
mean_li_mags=['phot_g_mean_mag_gaia_']

comp_df[dp2_offset_columns+mean_li_mags]